# 03 · Story agent (Task 1)

Runs the ReAct story agent on the Qdrant index built by `02_build_index.ipynb`, with the LLM behind an
**OpenAI-compatible endpoint** (llama.cpp `llama-server --jinja`, vLLM, Ollama, LM Studio, or a hosted API).

What it loads, all from the project folder:
* `qdrant_data/`: the `story_chunks` and `stories` collections (path set in `configs/retrieval.yaml`)
* `data/story_metadata.json`: titles and genres used for fuzzy matching (written by `prepare_data`)
* `nomic-ai/modernbert-embed-base` and FastEmbed BM25, to embed queries (local, not through the endpoint)

The agent writes a `Thought:`, calls a tool, reads the result, and repeats until it can answer.
Each example below prints that trace, the answer and any full story it fetched.

The endpoint must support **tool calling** (`tools` in `/chat/completions`). For llama.cpp, start the server with `--jinja`.

## 1. Endpoint settings

In [ ]:
import os

BASE_URL = os.environ.get("LLM_BASE_URL", "http://localhost:8080/v1")   # ends with /v1
MODEL = os.environ.get("LLM_MODEL", "qwen3-4b-instruct")               # the name your server expects
API_KEY = os.environ.get("LLM_API_KEY") or os.environ.get("OPENAI_API_KEY")   # None for local servers

# Optional: a different model (or endpoint) for summarize_story. None = same as the agent.
SUMMARY_MODEL = os.environ.get("LLM_SUMMARY_MODEL")
SUMMARY_BASE_URL = os.environ.get("LLM_SUMMARY_BASE_URL") or BASE_URL

# HTTPS: certificates are checked against the OS certificate store (truststore). Behind a company proxy or with a
# self-signed endpoint, point this at the CA's PEM file if the OS store does not already trust it.
CA_BUNDLE = os.environ.get("LLM_CA_BUNDLE")

TEMPERATURE = 0.0       # greedy: tool choice and arguments stay reproducible
MAX_NEW_TOKENS = 768

In [ ]:
# Quick check that the endpoint answers and supports tools, before loading anything heavy
from storybot.chat import OpenAICompatLLM

llm = OpenAICompatLLM(BASE_URL, MODEL, api_key=API_KEY, max_new_tokens=MAX_NEW_TOKENS, temperature=TEMPERATURE,
                      ca_bundle=CA_BUNDLE)
ping_tool = [{"type": "function", "function": {"name": "ping", "description": "Reply to a ping.",
              "parameters": {"type": "object", "properties": {}, "required": []}}}]
t = llm.chat([{"role": "user", "content": "Call the ping tool."}], tools=ping_tool)
print("text:", t.content or "-", "| tool calls:", [c.name for c in t.tool_calls] or "none (check tool calling on the server)")

## 2. Load the index and build the agent

In [ ]:
from storybot.chat import build_agent
from storybot.config import PROJECT_ROOT, load_config, resolve
from storybot.retrieval import index_is_built, open_client

rcfg = load_config()                                         # configs/retrieval.yaml
ccfg = load_config(PROJECT_ROOT / "configs" / "chat.yaml")   # agent + tool settings

client = open_client(rcfg)
assert index_is_built(rcfg, client), "no finished index: run 02_build_index.ipynb (or scripts/build_index.py) first"
for name in (rcfg.qdrant.chunks_collection, rcfg.qdrant.stories_collection):
    print(f"{name}: {client.count(name, exact=True).count} points")
print("metadata:", rcfg.data.metadata_path)

In [ ]:
agent = build_agent(retrieval_cfg=rcfg, chat_cfg=ccfg, llm=llm, client=client)   # loads ModernBERT + BM25 for queries

if SUMMARY_MODEL:
    agent.tools.llm = OpenAICompatLLM(SUMMARY_BASE_URL, SUMMARY_MODEL, api_key=API_KEY,
                                      max_new_tokens=MAX_NEW_TOKENS, temperature=TEMPERATURE,
                                      ca_bundle=CA_BUNDLE)

tools, matcher = agent.tools, agent.tools.matcher
print(len(matcher.title_map), "titles,", len(matcher.genres), "genres")
print("tools:", ", ".join(tools.names))

## 3. Pick example stories from the data

Examples use real ids, titles and genres from `story_metadata.json`, so the notebook works on any build
(including a `--limit` build: only indexed stories are picked).

In [ ]:
import random

indexed = {p.payload["story_id"] for p in client.scroll(rcfg.qdrant.stories_collection, limit=10_000,
                                                          with_payload=["story_id"])[0]}
entries = [v for v in matcher.title_map.values() if v["id"] in indexed]
rng = random.Random(7)
ex_a, ex_b, ex_c = rng.sample(entries, 3)
genre = ex_c["genre"]

def typo(s: str) -> str:
    """Swap two neighbouring letters in the longest word: a realistic typo for the fuzzy matcher."""
    w = max(s.split(), key=len)
    i = len(w) // 2
    return s.replace(w, w[:i - 1] + w[i] + w[i - 1] + w[i + 1:], 1) if len(w) > 3 else s.lower()

for e in (ex_a, ex_b, ex_c):
    print(e)
print("title with a typo:", typo(ex_b["title"]))

## 4. Matching and tools without the LLM

The tools are plain functions, so the fuzzy matching and fallbacks can be checked directly.

In [ ]:
import json

def show_json(d, n=1200):
    s = json.dumps(d, ensure_ascii=False, indent=1)
    print(s[:n] + (" ..." if len(s) > n else ""))

m = matcher.match_title(typo(ex_b["title"]))
print("title match:", m.value, round(m.score, 1), "| candidates:", [(c["title"], c["score"]) for c in m.candidates[:3]])
print("unknown title:", matcher.match_title("The Moon That Forgot To Shine").matched)
for g in (genre.lower(), typo(genre), "sci fi", "cooking"):
    mg = matcher.match_genre(g)
    print(f"genre {g!r:24} ->", mg.value, round(mg.score, 1))

In [ ]:
out = tools.search_stories("a journey to find something precious", genre=typo(genre))
print("genre filter:", out.get("genre_filter"), "| note:", out.get("genre_note"))
for r in out["results"]:
    print(r["story_id"], r["title"], "|", r["genre"], "|", r["score"])

In [ ]:
show_json(tools.get_story_by_title("The Moon That Forgot To Shine"))   # no title match -> semantic fallback

## 5. The agent on different kinds of questions

`ask()` runs one question and prints the trace (Thought / Action / Observation), the answer, the time,
and the start of any full story the agent fetched. Results are also collected for the summary at the end.

In [ ]:
import time

RUNS = []

def ask(question: str, show_story_chars: int = 600, trace: bool = True):
    t0 = time.time()
    res = agent.run(question)
    dt = time.time() - t0
    print(f"Q: {question}\n")
    if trace and res.steps:
        print(res.trace(), "\n")
    print("A:", res.answer)
    for st in res.stories:
        text = st["text"]
        print(f"\n--- {st['title']} (id {st['story_id']}, {st['genre']}) ---\n"
              f"{text[:show_story_chars]}{' ...' if len(text) > show_story_chars else ''}")
    print(f"\n[{len(res.steps)} tool calls, {dt:.1f}s]")
    RUNS.append({"question": question, "tools": [(s.tool, s.arguments) for s in res.steps],
                 "answer": res.answer, "seconds": round(dt, 1)})
    return res

### 5.1 Story details by id
Expected: `get_story_by_id`.

In [ ]:
agent.reset()
ask(f"Show me story {ex_a['id']}.")

### 5.2 Follow-up on the same story
Expected: `summarize_story` with the id from the previous turn.

In [ ]:
ask("Summarize it in a few sentences.")

### 5.3 Story details by title, with a typo
Expected: `get_story_by_title`; the fuzzy match finds the original title.

In [ ]:
agent.reset()
ask(f"Can you get me the story \"{typo(ex_b['title'])}\"?")

### 5.4 A title that does not exist
Expected: `get_story_by_title` finds no title and falls back to semantic search; the agent says so.

In [ ]:
agent.reset()
ask("Give me the story called 'The Moon That Forgot To Shine'.")

### 5.5 Several stories on a theme
Expected: `search_stories` with `num_stories` >= 6, all different stories.

In [ ]:
agent.reset()
ask("Recommend some stories about friendship and loyalty.")

### 5.6 Theme within a genre (genre typed with a typo)
Expected: `search_stories` with the genre fuzzy-matched to a real genre and used as a filter.

In [ ]:
agent.reset()
ask(f"I want {typo(genre).lower()} stories about overcoming fear.")

### 5.7 A genre that is not in the library
Expected: no genre match, so the search runs over all genres (see `genre_note`).

In [ ]:
agent.reset()
ask("Find me some cooking-show stories about a competition.")

### 5.8 A question about a character in one story
Expected: `ask_about_story` on ONE story; no passages from other stories.

In [ ]:
agent.reset()
ask(f"In the story \"{ex_c['title']}\", who is the main character and what do they want?")

### 5.9 Finding one story from a description
Expected: `search_stories` with a small `num_stories`, then the best match (often followed by `ask_about_story`).

In [ ]:
agent.reset()
hint = tools.retriever.get_story(ex_a["id"])["text"][:300]
ask(f"There is a story that starts roughly like this: \"{hint[:160]}...\" Which story is it, and how does it end?")

### 5.10 Detailed summary by title
Expected: `summarize_story` with `length=detailed`.

In [ ]:
agent.reset()
ask(f"Give me a detailed summary of \"{ex_c['title']}\".")

### 5.11 Browsing genres
Expected: `list_genres`, then `list_stories_by_genre`.

In [ ]:
agent.reset()
ask("Which genres do you have?", trace=False)
ask(f"List the stories in the {genre} genre.")

### 5.12 Out of scope
Expected: no tool call (or a search that finds nothing relevant); the agent says it only answers about the story library.

In [ ]:
agent.reset()
ask("What is the capital of France?")

## 6. Summary of the runs

In [ ]:
import pandas as pd

summary = pd.DataFrame([{"question": r["question"][:70],
                         "tools": " -> ".join(t for t, _ in r["tools"]) or "(none)",
                         "seconds": r["seconds"]} for r in RUNS])
pd.set_option("display.max_colwidth", 80)
summary

In [ ]:
out_path = resolve(rcfg, "results/agent_examples.json")
out_path.parent.mkdir(parents=True, exist_ok=True)
out_path.write_text(json.dumps({"model": MODEL, "base_url": BASE_URL, "runs": RUNS}, ensure_ascii=False, indent=1))
print("saved", out_path)

## 7. Test cases (`eval/agent_test_cases.json`)

51 cases built from the 15 stories in `eval/stories_sample.csv` (all part of the full dataset): lookups by id and title
(exact, typo, partial, missing), theme searches, genre filters with typos, finding one story from a description,
questions about one story (including names like Sarah, Jack and Victor that appear in several stories), summaries,
follow-ups, browsing and out-of-scope questions. Each case lists what the trace and the answer must show; see the
`fields` block at the top of the file. Set `CASE_IDS` to run a few cases only.

In [ ]:
from storybot.eval import load_cases, run_eval, summarize

CASE_IDS = None            # e.g. ["ask-15", "ask-16", "title-05"]
CATEGORIES = None          # e.g. ["ask_about_story", "find_one"]

cases = load_cases(ids=CASE_IDS, categories=CATEGORIES)
eval_rows = run_eval(agent, cases)
summarize(eval_rows)

In [ ]:
results = pd.DataFrame(eval_rows)
results["tools"] = results["tools"].map(lambda ts: " -> ".join(t for t, _ in ts) or "(none)")
failed = results[~results.passed][["id", "question", "failed_checks", "tools", "answer"]]
print(f"{results.passed.sum()}/{len(results)} passed")
failed

In [ ]:
eval_path = resolve(rcfg, "results/agent_eval.json")
eval_path.write_text(json.dumps({"model": MODEL, "summary": summarize(eval_rows), "cases": eval_rows},
                                ensure_ascii=False, indent=1))
print("saved", eval_path)